# In Class Activity April 14th, 2026

In [29]:
# pip install optuna , lightgbm , catboost

### Importing libraries, preparing data, initial EDA

In [51]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna
import lightgbm as lgb
from catboost import CatBoostClassifier


In [31]:
# importing data
adult = pd.read_csv('/Users/darrenbarkins/MSBA/ADML/data/adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [32]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





From the EDA, I seen that:
- The dataset has significant (class imbalance) . 
- Many columns contain (?) , but will be changed to (NaN)
- There's features that are categorical so boosting models like (XGBoost, LightGBM, Catboost) would would well because they can built in categorical handling

### Data Preprocessing (minimal) and Baseline Model

In [33]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

/var/folders/6w/pjn5hnkd5712p8r0ytbwcnt40000gn/T/ipykernel_93327/2764229057.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = adult.select_dtypes(include='object').columns


,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [38]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

# stratified k-fold for cross validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# class imbalance ratio — used for scale_pos_weight in XGBoost
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f'Class distribution — 0: {neg}, 1: {pos}')
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

Class distribution — 0: 29724, 1: 9349
scale_pos_weight: 3.18


In [39]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


---
## Baseline Model: XGBoost

In [36]:
# building xgboost model and evaluating k-fold cross validation

xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'XGBoost Baseline Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean():.4f}')

XGBoost Baseline Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7120


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

The default xgboost model reached a mean F1 score of ~0.71 using 5-fold stratified cross validation. Good starting point for class imbalance. To improve:

- Use `scale_pos_weight` to correct class imbalance
- Tune depth and learning rate to lower overfitting
- Still need to compare against Random Forest, LightGBM, and CatBoost

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [40]:
# your code here

# scale_pos_weight (handling class imbalance) 
print('=== Exploring scale_pos_weight ===')
for spw in [1, scale_pos_weight, scale_pos_weight * 2]:
    model = XGBClassifier(enable_categorical=True, random_state=42, scale_pos_weight=spw)
    scores = cross_val_score(model, X, y, cv=skf, scoring='f1')
    print(f'  scale_pos_weight={spw:.2f}  -> Mean F1: {scores.mean():.4f}')

print()

# max_depth (model complexity)
print('=== Exploring max_depth ===')
for depth in [3, 5, 7, 10]:
    model = XGBClassifier(enable_categorical=True, random_state=42,
                          scale_pos_weight=scale_pos_weight, max_depth=depth)
    scores = cross_val_score(model, X, y, cv=skf, scoring='f1')
    print(f'  max_depth={depth}  -> Mean F1: {scores.mean():.4f}')

print()

# learning_rate (step size / shrinkage)
print('=== Exploring learning_rate ===')
for lr in [0.01, 0.1, 0.2, 0.3]:
    model = XGBClassifier(enable_categorical=True, random_state=42,
                          scale_pos_weight=scale_pos_weight, learning_rate=lr)
    scores = cross_val_score(model, X, y, cv=skf, scoring='f1')
    print(f'  learning_rate={lr}  -> Mean F1: {scores.mean():.4f}')

print()

# Best identified config from exploration
print('=== Best config from exploration: scale_pos_weight + max_depth=5 + learning_rate=0.2 ===')
xgb_best_manual = XGBClassifier(enable_categorical=True, random_state=42,
                                 scale_pos_weight=scale_pos_weight, max_depth=5, learning_rate=0.2)
scores = cross_val_score(xgb_best_manual, X, y, cv=skf, scoring='f1')
print(f'  Mean F1: {scores.mean():.4f}')

=== Exploring scale_pos_weight ===
  scale_pos_weight=1.00  -> Mean F1: 0.7120
  scale_pos_weight=3.18  -> Mean F1: 0.7146
  scale_pos_weight=6.36  -> Mean F1: 0.6882

=== Exploring max_depth ===
  max_depth=3  -> Mean F1: 0.7123
  max_depth=5  -> Mean F1: 0.7150
  max_depth=7  -> Mean F1: 0.7140
  max_depth=10  -> Mean F1: 0.7122

=== Exploring learning_rate ===
  learning_rate=0.01  -> Mean F1: 0.6815
  learning_rate=0.1  -> Mean F1: 0.7137
  learning_rate=0.2  -> Mean F1: 0.7160
  learning_rate=0.3  -> Mean F1: 0.7146

=== Best config from exploration: scale_pos_weight + max_depth=5 + learning_rate=0.2 ===
  Mean F1: 0.7159


### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [41]:
# your code here

# GridSearchCV for parameter grid
# Tuning 4 hyperparameters: scale_pos_weight, max_depth, learning_rate, n_estimators

param_grid = {
    'scale_pos_weight': [1, scale_pos_weight],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.2, 0.3],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0]
}

xgb_grid = XGBClassifier(enable_categorical=True, random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_grid,
    param_grid=param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'Best parameters from GridSearchCV: {grid_search.best_params_}')
print(f'Best CV F1 score: {grid_search.best_score_:.4f}')

# evaluate best model on test set
y_pred_grid = grid_search.best_estimator_.predict(X_test)
print(f'\nClassification report for GridSearchCV-tuned model:')
print(classification_report(y_test, y_pred_grid))


Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best parameters from GridSearchCV: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'scale_pos_weight': np.float64(3.1793774735265803), 'subsample': 0.8}
Best CV F1 score: 0.7168

Classification report for GridSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.94      0.84      0.89      7431
           1       0.63      0.84      0.72      2338

    accuracy                           0.84      9769
   macro avg       0.78      0.84      0.80      9769
weighted avg       0.87      0.84      0.85      9769



### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [42]:
# tuning xgboost classifier with RandomizedSearchCV (tune more parameters than shown here)
param_dist = {
    'scale_pos_weight': np.linspace(1.0, 10.0, num=10),
    'max_depth': np.arange(3, 11),
    'learning_rate': np.linspace(0.01, 0.3, num=10)
}

# replace this placeholder model with your preferred model from above

xgb_random = RandomizedSearchCV(XGBClassifier(random_state=42, enable_categorical=True),
                                param_distributions=param_dist, n_iter=20, cv=skf, scoring='f1', random_state=42)
xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')   

# build your preferred model from above using best parameters from your RandomizedSearchCV
# and evaluate on the test set

xgb_random_best = XGBClassifier(random_state=42, scale_pos_weight=xgb_random.best_params_['scale_pos_weight'], 
                                max_depth=xgb_random.best_params_['max_depth'], 
                                learning_rate=xgb_random.best_params_['learning_rate'], 
                                enable_categorical=True)
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')


Best parameters from RandomizedSearchCV: {'scale_pos_weight': np.float64(2.0), 'max_depth': np.int64(9), 'learning_rate': np.float64(0.23555555555555557)}
Best F1 score from RandomizedSearchCV: 0.7157810076446193
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.92      0.89      0.91      7431
           1       0.69      0.76      0.72      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.83      0.81      9769
weighted avg       0.87      0.86      0.86      9769



### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [43]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    
    # replace this placeholder model with your preferred model from above
    
    xgb_optuna = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight, 
                               max_depth=max_depth,  learning_rate=learning_rate, enable_categorical=True)
    
    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=42, scale_pos_weight=study.best_params['scale_pos_weight'], 
                                  max_depth=study.best_params['max_depth'], 
                                  learning_rate=study.best_params['learning_rate'], 
                                  enable_categorical=True)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


[I 2026-04-15 23:11:56,653] A new study created in memory with name: no-name-65249091-3bb0-4523-a2d7-8d058ffe98a0
Best trial: 0. Best value: 0.682921:   5%|▌         | 1/20 [00:00<00:13,  1.46it/s]

[I 2026-04-15 23:11:57,341] Trial 0 finished with value: 0.682921248106645 and parameters: {'scale_pos_weight': 5.744079170433538, 'max_depth': 4, 'learning_rate': 0.21637850914490517}. Best is trial 0 with value: 0.682921248106645.


Best trial: 1. Best value: 0.69124:  10%|█         | 2/20 [00:01<00:10,  1.65it/s] 

[I 2026-04-15 23:11:57,889] Trial 1 finished with value: 0.69124049564857 and parameters: {'scale_pos_weight': 4.782156630607167, 'max_depth': 3, 'learning_rate': 0.26416095732672457}. Best is trial 1 with value: 0.69124049564857.


Best trial: 1. Best value: 0.69124:  15%|█▌        | 3/20 [00:02<00:16,  1.02it/s]

...

[I 2026-04-15 23:12:15,746] Trial 17 finished with value: 0.7044486214453107 and parameters: {'scale_pos_weight': 4.095558114877404, 'max_depth': 5, 'learning_rate': 0.2507356349819944}. Best is trial 16 with value: 0.7288360791034334.


Best trial: 16. Best value: 0.728836:  95%|█████████▌| 19/20 [00:19<00:00,  1.09it/s]

[I 2026-04-15 23:12:16,521] Trial 18 finished with value: 0.7261936680940636 and parameters: {'scale_pos_weight': 2.0915286172574494, 'max_depth': 5, 'learning_rate': 0.29578848048257594}. Best is trial 16 with value: 0.7288360791034334.


Best trial: 16. Best value: 0.728836: 100%|██████████| 20/20 [00:20<00:00,  1.03s/it]

[I 2026-04-15 23:12:17,297] Trial 19 finished with value: 0.7065083878955777 and parameters: {'scale_pos_weight': 3.9219914960521667, 'max_depth': 5, 'learning_rate': 0.29095714427556596}. Best is trial 16 with value: 0.7288360791034334.
Best parameters from Optuna: {'scale_pos_weight': 1.973395768513369, 'max_depth': 5, 'learning_rate': 0.2395565899647428}
Best F1 score from Optuna: 0.7288360791034334
Classification report for Optuna-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.88      0.91      7431
           1       0.68      0.78      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.83      0.82      9769
weighted avg       0.87      0.86      0.86      9769



### Random Forest
- Core approach: Bagging 
- Tree growth: Level-wise, balanced splits
- Categorical handling: Requires manual encoding 
- Class imbalance: class_weight='balanced'
- Best use case: Small/noisy data, strong baseline

In [46]:
# Random Forest requires numeric input, encode categoricals
X_rf = X.copy()
for col in X_rf.select_dtypes(include='category').columns:
    X_rf[col] = X_rf[col].cat.codes  # -1 for NaN, which RF handles fine

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf, y, test_size=0.2, shuffle=True, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced',  # handles class imbalance
    n_jobs=-1
)
rf.fit(X_train_rf, y_train_rf)
y_pred_rf = rf.predict(X_test_rf)

print('=== Random Forest ===')
print(classification_report(y_test_rf, y_pred_rf))

# cross-validated F1
rf_cv = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf_scores = cross_val_score(rf_cv, X_rf, y, cv=skf, scoring='f1')
print(f'Random Forest CV Mean F1: {rf_scores.mean():.4f}')

=== Random Forest ===
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      7431
           1       0.76      0.61      0.68      2338

    accuracy                           0.86      9769
   macro avg       0.82      0.77      0.79      9769
weighted avg       0.85      0.86      0.85      9769

Random Forest CV Mean F1: 0.6750


### LightGBM
- Core approach: Boosting (sequential ensemble of weak learners)
- Tree growth: Leaf-wise (best-first) — grows the leaf with the most loss reduction
- Categorical handling: Supported via dtype; best to specify explicitly
- Class imbalance: is_unbalance=True (automatic) or class_weight
- Speed: Fastest training on large datasets
- Overfitting risk: Higher — needs tuning (e.g. num_leaves, min_child_samples)

In [49]:
# LightGBM
cat_feature_names = X.select_dtypes(include='category').columns.tolist()

lgbm = lgb.LGBMClassifier(
    n_estimators=200,
    random_state=42,
    is_unbalance=True,       # handles class imbalance automatically
    num_leaves=31,           # controls complexity, smaller = less overfitting
    learning_rate=0.1,
    n_jobs=-1,
    verbose=-1
)
lgbm.fit(
    X_train, y_train,
    categorical_feature=cat_feature_names
)
y_pred_lgbm = lgbm.predict(X_test)

print('=== LightGBM ===')
print(classification_report(y_test, y_pred_lgbm))

# cross-validated F1
lgbm_cv = lgb.LGBMClassifier(
    n_estimators=100, random_state=42, is_unbalance=True, verbose=-1, n_jobs=-1
)
lgbm_scores = cross_val_score(lgbm_cv, X, y, cv=skf, scoring='f1')
print(f'LightGBM CV Mean F1: {lgbm_scores.mean():.4f}')

=== LightGBM ===
              precision    recall  f1-score   support

           0       0.95      0.83      0.89      7431
           1       0.61      0.86      0.72      2338

    accuracy                           0.84      9769
   macro avg       0.78      0.84      0.80      9769
weighted avg       0.87      0.84      0.85      9769

LightGBM CV Mean F1: 0.7139


### CatBoost

In [56]:
X_cat = X.copy()

# fix categorical columns — must be clean strings, no NaN
for col in X_cat.select_dtypes(include='category').columns:
    X_cat[col] = X_cat[col].astype(object).fillna('missing').astype(str)

# fix numeric columns — fill NaN with median
for col in X_cat.select_dtypes(include='number').columns:
    X_cat[col] = X_cat[col].fillna(X_cat[col].median())

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_cat, y, test_size=0.2, shuffle=True, random_state=42, stratify=y
)

cat_features_idx = [X_cat.columns.get_loc(c) for c in cat_feature_names]
cb_class_weights = [1, scale_pos_weight]

catboost = CatBoostClassifier(
    iterations=200,
    random_seed=42,
    class_weights=cb_class_weights,
    learning_rate=0.1,
    depth=6,
    verbose=0
)
catboost.fit(X_train_cat, y_train_cat, cat_features=cat_features_idx)
y_pred_cb = catboost.predict(X_test_cat)

print('=== CatBoost ===')
print(classification_report(y_test_cat, y_pred_cb))
print(f'CatBoost Test F1: {f1_score(y_test_cat, y_pred_cb):.4f}')

=== CatBoost ===
              precision    recall  f1-score   support

           0       0.95      0.82      0.88      7431
           1       0.60      0.86      0.71      2338

    accuracy                           0.83      9769
   macro avg       0.78      0.84      0.80      9769
weighted avg       0.87      0.83      0.84      9769

CatBoost Test F1: 0.7118


---
## Model Comparison Summary

In [58]:
# Summary table comparing all models
catboost_test_f1 = f1_score(y_test_cat, y_pred_cb)

results = {
    'Model': ['Random Forest', 'XGBoost (default)', 'XGBoost (Optuna-tuned)', 'LightGBM', 'CatBoost'],
    'F1 Score': [
        rf_scores.mean(),
        cv_scores.mean(),
        study.best_value,
        lgbm_scores.mean(),
        catboost_test_f1
    ],
    'Score Type': ['CV Mean F1', 'CV Mean F1', 'CV Mean F1', 'CV Mean F1', 'Test F1'],
    'Approach': ['Bagging', 'Boosting', 'Boosting', 'Boosting', 'Boosting'],
    'Categorical Handling': ['Manual (ordinal)', 'Built-in', 'Built-in', 'Auto-detect', 'Fully automatic'],
    'Imbalance Handling': ['class_weight=balanced', 'scale_pos_weight', 'scale_pos_weight (tuned)', 'is_unbalance=True', 'class_weights']
}

results_df = pd.DataFrame(results)
results_df['F1 Score'] = results_df['F1 Score'].round(4)
results_df = results_df.sort_values('F1 Score', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

                 Model  F1 Score Score Type Approach Categorical Handling       Imbalance Handling
XGBoost (Optuna-tuned)    0.7288 CV Mean F1 Boosting             Built-in scale_pos_weight (tuned)
              LightGBM    0.7139 CV Mean F1 Boosting          Auto-detect        is_unbalance=True
     XGBoost (default)    0.7120 CV Mean F1 Boosting             Built-in         scale_pos_weight
              CatBoost    0.7118    Test F1 Boosting      Fully automatic            class_weights
         Random Forest    0.6750 CV Mean F1  Bagging     Manual (ordinal)    class_weight=balanced


### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


(GridSearchCV) performs an exhaustive search over all combinations of the parameter grid. This guarantees finding the best combination within the grid, but it is computationally expensive. The number of fits equals (grid size) × (CV folds). It works well when the search space is small and you have a limited set of candidate values.

(RandomizedSearchCV) randomly samples from parameter distributions rather than testing every combination. This allows a much wider and more continuous search space (e.g., real-valued distributions for learning rate) with a fixed number of iterations. It is faster than GridSearchCV and often finds equally good or better results when the search space is large.

(Optuna) uses a Bayesian/TPE (Tree-structured Parzen Estimator) strategy to intelligently suggest the next set of hyperparameters based on past trial results. It focuses sampling effort on promising regions of the search space, making it the most efficient of the three, especially with more trials or more parameters. It also supports early pruning of unpromising trials.

(In practice:) Optuna tends to produce the best results with the least wasted computation. RandomizedSearchCV is a good middle ground. GridSearchCV is best reserved for final, small grids near a known good region.